# HER — Hindsight Experience Replay (2017)

_Papers · Reinforcement Learning_

**When an episode fails to reach its goal, pretend the state you DID reach was the goal all along — relabel the failed transitions, and a sparse-reward failure becomes a labeled success the agent can learn from.**

---

This notebook is a practice scaffold for the **HER — Hindsight Experience Replay (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch (Colab)

In [ ]:
# torch is preinstalled in Colab; nothing to install.
import random
from collections import deque
import numpy as np
import torch
import torch.nn as nn

random.seed(0); np.random.seed(0); torch.manual_seed(0)
GAMMA = 0.98

# --- 0. Sanity-check the lesson's worked relabel (n=5, final strategy). ---
def bit_reward(s_next, goal):                 # r_g(s,a) = -[s' != goal]
    return 0.0 if np.array_equal(s_next, goal) else -1.0

g_real = np.array([1,1,1,1,1])                # the episode's REAL goal 11111
s_T    = np.array([0,1,1,0,1])                # the state actually reached 01101
g_prime = s_T.copy()                          # final strategy: g' = s_T
s_next = np.array([0,1,1,0,1])                # next state of the last transition 01101
print("worked example:  real-goal reward =", bit_reward(s_next, g_real),
      " hindsight reward r' =", bit_reward(s_next, g_prime))
# worked example:  real-goal reward = -1.0  hindsight reward r' = 0.0


# --- 1. Bit-flipping environment: state in {0,1}^n, action i flips bit i, sparse reward. ---
class BitFlip:
    def __init__(self, n): self.n = n
    def reset(self):
        self.s = np.random.randint(0, 2, self.n)
        self.g = np.random.randint(0, 2, self.n)
        while np.array_equal(self.s, self.g):           # ensure goal != start
            self.g = np.random.randint(0, 2, self.n)
        return self.s.copy(), self.g.copy()
    def step(self, a):
        self.s = self.s.copy(); self.s[a] ^= 1          # flip bit a
        done = np.array_equal(self.s, self.g)
        return self.s.copy(), (0.0 if done else -1.0), done


# --- 2. Goal-conditioned Q-network: input (s, g) of length 2n, one value per flippable bit. ---
class QNet(nn.Module):
    def __init__(self, n, hidden=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2*n, hidden), nn.ReLU(), nn.Linear(hidden, n))
    def forward(self, x): return self.net(x)            # x = concat(s, g)


def enc(s, g):                                          # build the (s, g) network input
    return torch.tensor(np.concatenate([s, g]), dtype=torch.float32)


# --- 3. Train DQN with or without HER on n-bit bit-flipping; return the success-rate curve. ---
def train(use_her, n=15, episodes=2000, eps=0.2, batch=128):
    random.seed(0); np.random.seed(0); torch.manual_seed(0)
    env = BitFlip(n)
    q = QNet(n); q_target = QNet(n); q_target.load_state_dict(q.state_dict())
    opt = torch.optim.Adam(q.parameters(), lr=1e-3)
    buf = deque(maxlen=1_000_000)
    succ, curve = deque(maxlen=200), []
    for ep in range(episodes):
        s, g = env.reset(); traj, solved = [], False
        for t in range(n):                              # episode length capped at n steps
            if random.random() < eps: a = random.randrange(n)
            else:
                with torch.no_grad(): a = int(q(enc(s, g)).argmax())
            s2, r, done = env.step(a)
            traj.append((s.copy(), a, r, s2.copy(), done))
            buf.append((s.copy(), g.copy(), a, r, s2.copy(), done))   # standard replay (real goal)
            s = s2
            if done: solved = True; break
        succ.append(1.0 if solved else 0.0)

        # ---- THE HER RELABEL (Algorithm 1, 'final' strategy) ----
        if use_her:
            g_prime = traj[-1][3].copy()                # g' = s_T, the achieved final state
            for (st, a, r, s2, d) in traj:
                r_prime = 0.0 if np.array_equal(s2, g_prime) else -1.0   # RECOMPUTE reward under g'
                d_prime = np.array_equal(s2, g_prime)
                buf.append((st.copy(), g_prime.copy(), a, r_prime, s2.copy(), d_prime))  # HER copy

        # ---- ordinary DQN regression on the buffer (real + relabeled copies) ----
        if len(buf) >= batch:
            for _ in range(8):
                b = random.sample(buf, batch)
                S, G, A, R, S2, D = zip(*b)
                X  = torch.tensor(np.concatenate([np.array(S),  np.array(G)], 1), dtype=torch.float32)
                X2 = torch.tensor(np.concatenate([np.array(S2), np.array(G)], 1), dtype=torch.float32)
                A = torch.tensor(A); R = torch.tensor(R, dtype=torch.float32); D = torch.tensor(D, dtype=torch.float32)
                q_sa = q(X).gather(1, A.unsqueeze(1)).squeeze(1)        # Q(s,a,g)
                with torch.no_grad():
                    y = (R + GAMMA * (1 - D) * q_target(X2).max(1).values).clamp(-50.0, 0.0)
                loss = (q_sa - y).pow(2).mean()
                opt.zero_grad(); loss.backward(); opt.step()
            q_target.load_state_dict(q.state_dict())     # sync target net
        if ep % 100 == 0:
            sr = sum(succ) / len(succ); curve.append((ep, round(sr, 3))); print(f"  ep {ep:4d}  success {sr:.2f}")
    return curve

print("\nDQN + HER (final strategy), n=15 bits:")
her_curve = train(use_her=True,  n=15)
print("\nABLATION -- vanilla DQN, NO HER (same net/buffer/opt/seed), n=15 bits:")
noher_curve = train(use_her=False, n=15)
print("\nHER   success curve:", her_curve[::4])
print("no-HER success curve:", noher_curve[::4])
# DQN+HER climbs toward ~0.9 success; vanilla DQN stays near 0 at n=15 (past the paper's
# n<=13 vanilla ceiling). Our small run, not the paper's numbers (paper: HER solves up to n=50).

## Visualize the data & results

_On the n=15 bit-flipping task with the sparse binary reward, does DQN+HER (relabel each failed episode with the achieved final state and store those copies) drive the success rate up, while vanilla DQN -- same network, buffer, optimizer, and seed, but NO relabeling -- stays stuck near 0? We train both for the same number of episodes and plot the rolling success rate._

In [ ]:
# Sketch of how the two curves above are produced (full loop in the CODE cell).
# Train DQN+HER and vanilla DQN on n=15 bit-flipping for the same episodes with identical
# goal-conditioned net / buffer / optimizer / lr / seed, recording rolling success rate.
#
#   her_curve   = train(use_her=True,  n=15)   # green: relabel g'=s_T, recompute r' -> climbs to ~0.9
#   noher_curve = train(use_her=False, n=15)   # red:   no relabel -> every reward is -1 -> stuck near 0
#
# The HER relabel is the only difference:
#   g_prime = traj[-1][3]                       # achieved final state s_T
#   r_prime = 0.0 if (s2 == g_prime).all() else -1.0   # RECOMPUTE the sparse reward under g'
#   buf.append((s||g_prime, a, r_prime, s2||g_prime, done'))   # store the hindsight copy
#
# DQN+HER -> success climbs toward ~0.9.  Vanilla DQN -> stays ~0 (no non-(-1) reward ever seen).
# (Numbers are from our own n=15 run; the paper reports HER solving up to n=50, not these values.)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The worked relabel. Bit-flipping with $n = 5$. An episode aimed at $g = 11111$ but ended at
            $s_T = 01101$. Consider its last transition: state $s = 01001$, action $a = 2$ (flip bit index $2$),
            next state $s' = 01101$. Using the final strategy, give the hindsight goal $g'$, the
            real-goal reward, and the recomputed hindsight reward $r'$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Hindsight goal (final): $g' = s_T = 01101$. — _The final strategy relabels with the state actually reached at the end of the episode (Algorithm 1, $\mathbb{S} = m(s_T)$)._
- Real-goal reward: $r_g(s,a) = -[s' \neq g] = -[01101 \neq 11111] = -1$. — _The next state is not the real goal $11111$, so the honest sparse reward is $-1$ — a failed step._
- Hindsight reward: $r' = -[s' \neq g'] = -[01101 \neq 01101] = -[\text{false}] = 0$. — _Under the relabeled goal, $s'$ exactly equals $g'$, so the Iverson bracket is $0$ and the reward is $0$ — a success._

**Answer:** $g' = 01101$; real-goal reward $= -1$; hindsight reward $r' = 0$. The physical step
                 $01001 \to 01101$ is unchanged; only the goal-tag flipped ($11111 \to 01101$), flipping the
                 reward ($-1 \to 0$). The notebook recomputes $-[01101\neq 11111] = -1$ and
                 $-[01101\neq 01101] = 0$.

</details>

**Problem 2.** Why off-policy makes it legal. A skeptic says "you are inventing fake experience by changing
            the goal — that must bias the learner." Explain why relabeling is unbiased for an off-policy RL
            algorithm.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify what a relabeled transition keeps fixed: the physical step $(s, a, s')$. — _The environment dynamics do not depend on the goal — flipping bit $i$ maps $s\to s'$ regardless of the target — so $(s,a,s')$ is a true fact for every goal._
- Identify what changes: the goal $g\to g'$ and hence the recomputed reward $r' = -[f_{g'}(s')=0]$. — _The goal enters the MDP only through the reward predicate $f_g$, so attaching $g'$'s reward to the same step yields a transition that is genuinely valid for goal $g'$._
- Invoke off-policy learning: the Bellman update uses stored $(s,a,r,s')$ regardless of which policy/goal produced them. — _Off-policy methods (DQN, DDPG) bootstrap from arbitrary stored transitions, so training on the relabeled copy is the same operation as training on a real goal-$g'$ episode._

**Answer:** It is unbiased because the dynamics are goal-independent: the physical step $(s,a,s')$
                 is a valid transition for every goal, and the goal only changes the reward via $f_g$.
                 Relabeling keeps $(s,a,s')$ and swaps in $(g', r')$, producing a transition that truly belongs
                 to goal $g'$. Since the learner is off-policy and trains on stored transitions irrespective of
                 their source, no bias is introduced — relabeling only adds the success signals the sparse reward
                 withheld.

</details>

**Problem 3.** The ablation. Your DQN+HER solves $n = 15$ bit-flipping (success rate climbs toward ~0.9).
            Run the ablation that removes only the relabeling step — same goal-conditioned network,
            buffer, optimizer, learning rate, and seed. Predict the success-rate curve and explain what it
            demonstrates.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Remove step 4 (the HER relabel): store only the real-goal transitions $(s\|g, a, r, s'\|g)$. — _Without relabeling, a random agent at $n=15$ almost never lands exactly on the target, so every stored transition carries reward $-1$._
- Train identically and record the rolling success rate. — _The Bellman target $y = -1 + \gamma\max_{a'}Q$ has no reward-contrast, so there is no gradient signal distinguishing good actions from bad._
- Compare to the full DQN+HER curve. — _Each run changes exactly one thing (the relabel), isolating HER as the cause._

**Answer:** The no-HER curve stays pinned near $0$ — at $n = 15$ (past the paper's $n\le 13$ vanilla
                 ceiling) the agent essentially never reaches the target by chance, so every reward is $-1$ and
                 there is nothing to learn from. DQN+HER climbs toward ~0.9 because relabeling manufactures the
                 first non-$(-1)$ rewards. Since the runs differ only in the relabel step, this isolates hindsight
                 relabeling as what makes sparse-reward learning possible. The CODEVIZ panel shows this contrast.

</details>